In [7]:
import os
import sys
import pandas as pd
import torch
from tqdm import tqdm
import numpy as np

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
)

from transformers import BertTokenizer, pipeline
import platform
from tqdm.auto import tqdm
from pathlib import Path

In [8]:
cwd = os.getcwd()
config_path = os.getcwd()

sys.path.append(config_path)
import config

database = config.database
central_banks = config.central_banks
training_data = os.path.join(database, "Training Data")
fed_docs = config.fed_docs
ecb_docs = config.ecb_docs

# hard coded paths for now, will need to be changed for different users before deployment
fed_training_data = pd.read_csv(
    "/Users/kylenabors/Documents/Database/Testing Data/Labeled Sentiment Randomized/Fed Labeled Sentiment.csv"
)
ecb_training_data = pd.read_csv(
    "/Users/kylenabors/Documents/Database/Testing Data/Labeled Sentiment Randomized/ECB Labeled Sentiment.csv"
)

In [9]:
platform.platform()

torch.backends.mps.is_built()

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print("MPS device not found.")

model = BertForSequenceClassification.from_pretrained(
    "ZiweiChen/FinBERT-FOMC", num_labels=3
).to("mps")

tokenizer = BertTokenizer.from_pretrained("ZiweiChen/FinBERT-FOMC")

finbert_fomc = pipeline(
    "text-classification", model=model, tokenizer=tokenizer, device="mps"
)

url_map = pd.read_csv(os.path.join(cwd, "url_map.csv"))

tensor([1.], device='mps:0')


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40273.97it/s]


In [10]:
class FinBERTDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        text = self.dataframe.iloc[index]["text"]
        label = self.dataframe.iloc[index]["label"]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [11]:
model = torch.load(os.path.join(cwd, "finbert_model.pt"))
tokenizer = torch.load(os.path.join(cwd, "finbert_tokenizer.pt"))

dataset = FinBERTDataset(
    training_data,
    tokenizer,
    max_len=128,
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kylenabors/Documents/GitHub/Central Bank FinBERT Analysis/finbert_model.pt'